Encode pen-lifting into the action space.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset, GmlDataset
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry

In [ ]:
pred_horizon = 512
obs_horizon = 1  # pad end but don't pad start, to discourage standing still at start
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 5  # dx/dy/penup

num_diffusion_iters = 100

## Load Dataset

In [ ]:
# dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"
dataset_path = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"

# create dataset from file
# dataset = PushTStateDataset(
#     dataset_path=dataset_path,
#     pred_horizon=pred_horizon,
#     obs_horizon=obs_horizon,
#     action_horizon=action_horizon,
#     action_delta=True
# )
with gerry.Stopwatch("Loading dataset"):
    dataset = GmlDataset(
        dataset_path=dataset_path,
        sequence_length=pred_horizon,
        pad_before=0,
        pad_after=0,
        # stride=10,
        action_delta=True,
        action_penlift=True,
        normalize=dict(obs=False, action=True),
        # max_drawings=100
    )
print(f'The number of drawings is {len(dataset.episode_ends)}')
print(dataset.indices.shape)
# print(dataset.episode_ends)
# print(dataset.indices)

# create dataloader
with gerry.Stopwatch("Creating dataloader"):
    dataloader = torch.utils.data.DataLoader(
        dataset,
        batch_size=128,
        num_workers=1,
        shuffle=True,
        pin_memory=True,
        persistent_workers=True
    )

# visualize data in batch
print("Num batches:          ", len(dataloader))
batch = next(iter(dataloader))
print("batch['obs'].shape:   ", batch['obs'].shape)
print("batch['action'].shape:", batch['action'].shape)

In [ ]:
acts = dataset.normalized_train_data['action'][:dataset.episode_ends[0]]
obss = dataset.normalized_train_data['obs'][:dataset.episode_ends[0]]
print(f'{acts.shape=}   {obss.shape=}')

fig, axes = plt.subplots(2, 1, figsize=(6, 5))
axes[0].plot(obss, '.-', linewidth=0.5)
axes[1].plot(acts, '.-', linewidth=0.5)
axes[1].set_ylim(-2, 2);

In [ ]:
def plot_traj(ax, action_n, obs=None, x0=[0, 0], travel_ls='k:', travel_kwargs=dict(), line_ls='.-', **line_kwargs):
    pen_up = action_n[:, 2] > 0.5
    action = dataset.unnormalize_action(action_n)
    
    if obs is None:
        obs = np.concatenate(([[0, 0]], np.cumsum(action[:, :2], axis=0))) + x0

    for i in np.argwhere(pen_up).flatten():
        travel = obs[i] + [[0, 0], action[i, :2].tolist()]
        ax.plot(*travel.T, travel_ls, **travel_kwargs)
    s = -1
    for i in np.argwhere(pen_up).flatten():
        ax.plot(*obs[s + 1:i + 1].T, line_ls, **line_kwargs)
        s = i
    ax.plot(*obs[s + 1:].T, line_ls, **line_kwargs)

plot_traj(plt, acts, x0=obss[0])
plt.axis('equal');

### Debug Visualization

In [ ]:
trajs = [dataset.normalized_train_data['obs'][dataset.episode_ends[10*k]:dataset.episode_ends[10*k+1]] for k in range(100)]
actions = [dataset.normalized_train_data['action'][dataset.episode_ends[10*k]:dataset.episode_ends[10*k+1]] for k in range(100)]

In [ ]:
fig, axes = plt.subplots(10, 10, figsize=(15, 15))
for ax, traj, action in zip(axes.flat, trajs, actions):
    ax.plot(traj[:, 0], traj[:, 1], 'k.-')
    # plt.quiver(traj[:-1, 0], traj[:-1, 1], action[:, 0], action[:, 1], scale=10)
fig.suptitle('Dataset Episodes')

In [ ]:
fig, axes = plt.subplots(len(batch.keys()), 1, figsize=(10, 6))

batch = next(iter(dataloader))
for i in range(10):
    obs = np.cumsum(dataset.unnormalize_action(batch['action'][i]), axis=0) * 3 + torch.tensor([1 * i, 0])
    obs = dataset.unnormalize_obs(batch['obs'][i]) * 3 + torch.tensor([1 * i, 0])
    axes[0].plot(*obs.T, '.-')
    axes[0].set_title('obs (unnormalized)')
    # axes[1].plot(*batch['action'][i].T, '.-')
    axes[1].plot(batch['action'][i], '-')
    axes[1].set_title('action (normalized)')
# for ax in axes:
#     ax.axis('equal')
axes[0].axis('equal');
fig.suptitle('Visualizing individual 128-len training samples')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i in range(1, 20):
    s, e = dataset.episode_ends[i - 1], dataset.episode_ends[i]
    axes[0].plot(*dataset.normalized_train_data['obs'][s:e].T, '.-')
    axes[0].axis('equal')
for i in range(1, len(dataset.episode_ends), len(dataset.episode_ends) // 100):
    s, e = dataset.episode_ends[i - 1], dataset.episode_ends[i]
    axes[1].plot(*dataset.normalized_train_data['obs'][s:e].T, '.-')
    axes[1].axis('equal')
fig.suptitle('Dataset episodes')

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
# Network!
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=0,
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
noise_pred_net.apply(init_weights);

In [ ]:
# Test with example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, obs_horizon, obs_dim))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# compute
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs.flatten(start_dim=1))

# check denoising
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

## Training

In [ ]:
num_epochs = 500 // len(dataloader) + 1
num_epochs = 1500 // len(dataloader) + 1

ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

if True:
    noise_pred_net.apply(init_weights)
all_losses = list()

writer = SummaryWriter()  # Tensorboard

In [ ]:
try:
    starting_step = len(all_losses)
    with tqdm(range(num_epochs), desc='Epoch') as tglobal:
        # epoch loop
        for epoch_idx in tglobal:
            epoch_loss = list()
            with tqdm(dataloader, desc='Batch') as tdataloader:
                # batch loop
                for batch_n in tdataloader:
                    # Extract data
                    obs_n = batch_n['obs'].to(device)
                    action_n = batch_n['action'].to(device)
                    action_n = torch.cat([obs_n, action_n], dim=-1)
                    B = obs_n.shape[0]
                    # assert obs_n.shape[1] == obs_horizon
                    # global_cond = obs_n.flatten(start_dim=1)
                    global_cond = None

                    # sample noise to add to actions
                    noise = torch.randn(action_n.shape, device=device)
                    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (B,), device=device).long()
                    noisy_actions = noise_scheduler.add_noise(action_n, noise, timesteps)

                    # predict the noise residual
                    noise_pred = noise_pred_net(noisy_actions, timesteps, global_cond=global_cond)

                    # L2 loss
                    loss = nn.functional.mse_loss(noise_pred, noise)

                    # optimize
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    lr_scheduler.step()

                    ema.step(noise_pred_net)

                    # logging
                    loss_cpu = loss.item()
                    tdataloader.set_postfix(loss=loss_cpu)
                    epoch_loss.append(loss_cpu)
                    writer.add_scalar('Loss', loss_cpu, global_step=starting_step + len(all_losses) + len(epoch_loss))

            tglobal.set_postfix(loss=np.mean(epoch_loss))
            all_losses.extend(epoch_loss)
            writer.add_scalar('Loss/Epoch', np.mean(epoch_loss), global_step=epoch_idx)
except KeyboardInterrupt:
    all_losses.extend(epoch_loss)
    pass
writer.close()

In [ ]:
# Weights of the EMA model is used for inference
ema_noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=obs_dim*obs_horizon,
)
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())
torch.save(ema_noise_pred_net.state_dict(), f'{writer.log_dir}/ema_noise_pred_net.pth')
# torch.jit.save(torch.jit.script(ema_noise_pred_net), f'{writer.log_dir}/ema_noise_pred_net.pt')

# Print the log directory where the weights are saved
print(writer.log_dir)

# Plot the loss
plt.figure(figsize=(10, 3))
plt.semilogy(all_losses)
plt.title('Loss')

## Inference

In [ ]:
if True:
    # load pretrained weights
    # This is default, training on 3000 drawings
    folder1 = f'runs/Apr04_21-01-28_eagle'
    # # training on 100 drawings
    # folder1 = f'runs/Apr05_16-14-37_eagle'
    # # training with 512-length trajectories
    # folder1 = 'runs/Apr05_16-48-06_eagle'

    device = torch.device('cuda')
    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
    )
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(f'{folder1}/ema_noise_pred_net.pth'))
    global_cond = None

In [ ]:
# B = 15  # num samples
B = 6*6  # num samples
all_obs = {}
all_actions = {}
all_histories = {}

# for horizon in tqdm([40, 80, 160, 320, 640, 1280, 2560]):
for horizon in tqdm([2560]):
    # action_n_init = torch.randn((B, pred_horizon, action_dim), device=device)
    action_n_init = torch.randn((B, horizon, action_dim), device=device)
    history = []

    action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                            global_cond=global_cond[[0]] if global_cond is not None else None,
                            log_history=history)

    action_n = action_n.detach().cpu().numpy()
    action = dataset.unnormalize_action(action_n[..., -3:])

    all_actions[horizon] = action
    all_histories[horizon] = history
    all_obs[horizon] = dataset.unnormalize_obs(action_n[..., :-3])

In [ ]:
# Compare states from diffusion vs feedforward from actions
fig = plt.figure(figsize=(15, 4), constrained_layout=True, facecolor=(0,0,0,0))
subfigs = fig.subfigures(1, len(all_obs), facecolor='white', wspace=0.1)

for subfig, k in zip(subfigs, sorted(all_obs.keys())):
    obss, acts = all_obs[k], all_actions[k]
    axes = subfig.subplots(2, 1, sharex=True, sharey=True)
    act = dataset.normalize_action(acts[0])
    plot_traj(axes[0], act, x0=obss[0][0])
    plot_traj(axes[1], act, obs=obss[0])
    axes[0].grid(False)
    axes[1].grid(False)
    subfig.suptitle(f'T = {k}')
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)

# fig.suptitle('Integrated diffusion delta (top) vs diffusion pos (bottom)');

In [ ]:
scale = 1
# Plot trajectories
r, c = (B - 1) // 6 + 1, 6
fig, axes = plt.subplots(r, c, figsize=(12 / scale, 2.5 * r / scale))
fig.subplots_adjust(hspace=0.1, wspace=0.1)
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
for act, obs, ax in zip(acts, obss, axes.flatten()):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset.normalize_action(act), x0=obs[0], markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
    # ax.axis('off')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle(f'T = {act.shape[0]}', fontsize=64 / scale);

In [ ]:
scale = 1
# Plot trajectories
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
selected = [12, 24, 26, 27, 30, 31, 32, 4, 2]
ranges = [-1, -1, -1, -1, -1, -1, 1500, -1, 1500]
for act, obs, ax, rangee in zip(acts[selected], obss[selected], axes.flatten(), ranges):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset.normalize_action(act[:rangee]), x0=obs[0], markersize=.5, linewidth=0.25, travel_kwargs=dict(linewidth=0.5), line_ls='k.-')
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
    # ax.axis('off')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Unconditioned DDPM Outputs', fontsize=32)
fig.set_tight_layout(True)
fig.savefig('results/figs/unconditioned_drawings_black.eps')
np.savez('results/figs/unconditioned_drawings_black.npz', acts=acts, obss=obss, selected=selected, ranges=ranges)

In [ ]:
# Plot trajectories
fig = plt.figure(constrained_layout=True, figsize=(4 * len(all_obs), 7), facecolor=(0, 0, 0, 0))

subfigs = fig.subfigures(nrows=1, ncols=len(all_obs), wspace=0.1, facecolor='white')
for subfig, acts in zip(subfigs, all_actions.values()):
    r, c = (B - 1) // 3 + 1, 3
    r, c = 4, 2
    # fig, axes = plt.subplots(r, c, figsize=(12, 2.5 * r))
    # fig, axes = plt.subplots(r, c, figsize=(4, 7))
    axes = subfig.subplots(r, c)
    print(acts.shape)
    for act, ax in zip(acts, axes.flatten()):
        # ax.plot(*(obs - obs[0]).T, 'k.-', markersize=2, linewidth=0.5)
        plot_traj(ax, dataset.normalize_action(act), x0=obss[0][0])
        ax.axis('equal')
        # ax.axis('off')
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])
        ax.grid(False)
        ax.set_xticks([])
        ax.set_yticks([])
    subfig.suptitle(f'T = {act.shape[0]}', fontsize=32)

## Visualize Denoising

In [ ]:
# Create animation
outfolder = Path('results/gerry09d_batch')
outfolder.mkdir(exist_ok=True)

def plot_frame(axes, actions_n, gt, t):
    for act_n, ax in zip(actions_n, axes.flatten()):
        ax.clear()
        # obs = res['obs']
        # ax.plot(*(obs - obs[0]).T, 'k.-')
        plot_traj(ax, act_n[:, -3:], x0=dataset.unnormalize_obs(act_n[0, :-3]))
        ax.axis('equal')


r, c = (B - 1) // 5 + 1, 5
fig, axes = plt.subplots(r, c, figsize=(12, 1.5 * r + 2))
fig.suptitle(f'T = 000\nt=0', fontsize=32)
fig.tight_layout()
for t, actions in enumerate(tqdm(all_histories[40 * 64])):
    plot_frame(axes, actions, gt, t)
    fig.suptitle(f'T = {actions[0].shape[0]}\nt = {t}', fontsize=32)
    fig.savefig(outfolder / f'frame_{t:03d}.png')

In [ ]:
# Create video
!/home/gchen328/miniconda3/bin/ffmpeg -y -r 25 -i results/gerry09d_batch/frame_%03d.png -c:v h264 -pix_fmt yuv420p results/gerry09d_batch.mp4 -hide_banner -loglevel error
Path('results/gerry09d_batch.mp4').rename(f'results/gerry09d_batch_{actions[0].shape[0]}.mp4')

In [ ]:
# Display
from IPython.display import Video
Video(f"results/gerry09d_batch_{actions[0].shape[0]}.mp4", width=512, height=256)

## Denoise a training sample

In [ ]:
t_start = 10

e = (dataset.episode_ends[0] // 8) * 8
obs_n = dataset.normalized_train_data['obs'][:e]
actions_n = dataset.normalized_train_data['action'][:e]

obs_n, actions_n = torch.tensor(obs_n), torch.tensor(actions_n)
x = torch.concatenate((obs_n, actions_n), dim=-1).to(device)[None, ...]

print(obs_n.shape, actions_n.shape, x.shape)

# Add initial noise
noise = torch.randn(x.shape, device=device)
noisy_x = noise_scheduler.add_noise(x, noise, torch.tensor([t_start]).to(device))

final_x = network.eval_partial(ema_noise_pred_net, noise_scheduler, noisy_x, t_start)

In [ ]:
if True:
    # load pretrained weights
    # This is default, training on 3000 drawings
    folder1 = f'runs/Apr04_21-01-28_eagle'
    # # training on 100 drawings
    # folder1 = f'runs/Apr05_16-14-37_eagle'
    # # training with 512-length trajectories
    # folder1 = 'runs/Apr05_16-48-06_eagle'

    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=obs_dim*obs_horizon,
    )
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(f'{folder1}/ema_noise_pred_net.pth'))
    global_cond = None

final_x = network.eval_partial(ema_noise_pred_net, noise_scheduler, noisy_x, t_start)

In [ ]:
def plot_x(ax, x, title):
    x = x.detach().cpu().numpy()
    obs = dataset.unnormalize_obs(x[0, :, :2])
    act = x[0, :, 2:]
    plot_traj(ax, act, x0=obs[0])
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))

plot_x(axes[0], x, 'Original Trajectory')
plot_x(axes[1], noisy_x, 'Noisy Trajectory')
plot_x(axes[2], final_x, 'Denoised Trajectory')

## Visualize Drawing

In [ ]:
# Create animation
outfolder = Path('results/gerry09d_batch')
outfolder.mkdir(exist_ok=True)

def plot_frame(axes, action_ns, obss, t):
    for ax, action_n, obs in zip(axes.flatten(), action_ns, obss):
        ax.clear()
        plot_traj(ax, action_n[:t, -3:], x0=obs[0])
    # ax.axis('equal')
    # ax.set_xlim(0, 1)
    # ax.set_ylim(0, 1)

for mult in [1, 16, 64]:
    actions = all_actions[40*mult]
    obss = all_obs[40*mult]
    actions = dataset.normalize_action(actions)
    action = dataset.normalize_action(actions[0]) * 0
    # obs = obss[0]

    !rm results/gerry09d_batch/aframe_*.png

    fig, axes = plt.subplots(1, 5, figsize=(12, 3))
    fig.suptitle(f'T = {action.shape[0]}', fontsize=24)

    # Get final x/y limits
    plot_frame(axes, actions, obss, len(action))
    for ax in axes.flatten():
        ax.axis('equal')
    xlims, ylims = [ax.get_xlim() for ax in axes], [ax.get_ylim() for ax in axes]

    for t in tqdm(range(2, action.shape[0])):
        # plot_frame(axes, action, obs, t)
        plot_frame(axes, actions, obss, t)
        for ax, xlim, ylim in zip(axes, xlims, ylims):
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.grid(False)
            ax.set_xticks([])
            ax.set_yticks([])
        # axes.set_title(f'T = {t + 1:4d} / {action.shape[0]}')
        fig.savefig(outfolder / f'aframe_{t:03d}.png')

    # Create video
    !/home/gchen328/miniconda3/bin/ffmpeg -y -r 25 -i results/gerry09d_batch/aframe_%03d.png -c:v h264 -pix_fmt yuv420p results/gerry09d_batch.mp4 -hide_banner -loglevel error
    Path('results/gerry09d_batch.mp4').rename(f'results/gerry09d_batch_watch_100draw_{actions[0].shape[0]}.mp4')

In [ ]:
# Display
from IPython.display import Video
Video(f"results/gerry09d_batch_watch_100draw_{actions[0].shape[0]}.mp4", width=512, height=256)